# 🚦 Traffic Demand Prediction — Optimized CatBoost Pipeline
> Target: **R² ≥ 0.98**

**Optimizations applied:**
- Log1p target transform for skewed demand
- Rich cyclical + interaction features
- Granular 15-min time slots
- Weather severity encoding
- Tuned CatBoost hyperparameters (depth=10, iter=8000)
- 10-Fold CV with early stopping

## 1. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# ── CONFIG ────────────────────────────────────────────────────
USE_LOG_TARGET = True   # Toggle: True usually gives best R² on skewed targets
N_SPLITS       = 10
SEED           = 42

print('✅ Imports done')
print(f'   USE_LOG_TARGET = {USE_LOG_TARGET}')
print(f'   N_SPLITS       = {N_SPLITS}')

## 2. Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
submission_ids = test['Index']

print(f'Train shape : {train.shape}')
print(f'Test  shape : {test.shape}')
train.head()

In [ ]:
# Quick EDA
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train['demand'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Demand Distribution (raw)')
axes[0].set_xlabel('demand')

axes[1].hist(np.log1p(train['demand']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Demand Distribution (log1p)')
axes[1].set_xlabel('log1p(demand)')

plt.tight_layout()
plt.show()

print('\nMissing values in train:')
print(train.isnull().sum()[train.isnull().sum() > 0])

## 3. Preprocessing — Fill Missing Values

In [ ]:
for col in ['RoadType', 'Weather']:
    train[col] = train[col].fillna('Unknown')
    test[col]  = test[col].fillna('Unknown')

temp_median = train['Temperature'].median()
train['Temperature'] = train['Temperature'].fillna(temp_median)
test['Temperature']  = test['Temperature'].fillna(temp_median)

print(f'Temperature median used for imputation: {temp_median:.2f}')
print('✅ Missing values handled')

## 4. Feature Engineering

In [ ]:
# Weather severity numerical mapping
WEATHER_SEVERITY = {
    'Clear': 0, 'Sunny': 0,
    'Cloudy': 1, 'Overcast': 1,
    'Fog': 2, 'Mist': 2,
    'Rain': 3, 'Drizzle': 3,
    'HeavyRain': 4, 'Snow': 4, 'Storm': 5,
    'Unknown': 1
}

ROAD_ORDER = {
    'Local': 0, 'Arterial': 1,
    'Highway': 2, 'Expressway': 3,
    'Unknown': 1
}

def feature_engineering(df):
    df = df.copy()
    ts = df['timestamp'].astype(str).str.split(':', expand=True)
    df['hour']   = ts[0].astype(int)
    df['minute'] = ts[1].astype(int)

    # ── Cyclical time features ─────────────────────────────
    df['hour_sin']   = np.sin(2 * np.pi * df['hour']   / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour']   / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)

    # ── Time-of-day buckets ────────────────────────────────
    df['is_morning']    = ((df['hour'] >= 6)  & (df['hour'] <= 11)).astype(int)
    df['is_afternoon']  = ((df['hour'] >= 12) & (df['hour'] <= 16)).astype(int)
    df['is_evening']    = ((df['hour'] >= 17) & (df['hour'] <= 21)).astype(int)
    df['is_night']      = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    df['is_late_night'] = ((df['hour'] >= 0)  & (df['hour'] <= 4)).astype(int)
    df['is_peak_hour']  = (
        ((df['hour'] >= 7)  & (df['hour'] <= 10)) |
        ((df['hour'] >= 17) & (df['hour'] <= 20))
    ).astype(int)

    # ── Polynomial / interaction ───────────────────────────
    df['lane_hour']   = df['NumberofLanes'] * df['hour']
    df['lane_square'] = df['NumberofLanes'] ** 2
    df['hour_square'] = df['hour'] ** 2
    df['hour_cube']   = df['hour'] ** 3

    # ── Temperature features ───────────────────────────────
    df['temp_lane']    = df['Temperature'] * df['NumberofLanes']
    df['temp_hour']    = df['Temperature'] * df['hour']
    df['temp_peak']    = df['Temperature'] * df['is_peak_hour']
    df['temp_squared'] = df['Temperature'] ** 2

    # ── Weather / road numericals ──────────────────────────
    df['weather_severity'] = df['Weather'].map(WEATHER_SEVERITY).fillna(1)
    df['road_order']       = df['RoadType'].map(ROAD_ORDER).fillna(1)
    df['severity_lane']    = df['weather_severity'] * df['NumberofLanes']
    df['severity_peak']    = df['weather_severity'] * df['is_peak_hour']

    # ── Granular time slot (15-min buckets → 96 slots/day) ─
    df['minute_bucket'] = (df['minute'] // 15).astype(int)
    df['time_of_day']   = df['hour'] * 4 + df['minute_bucket']

    # ── Cross categorical features ─────────────────────────
    df['road_lane']    = df['RoadType'].astype(str) + '_' + df['NumberofLanes'].astype(str)
    df['day_hour']     = df['day'].astype(str)      + '_' + df['hour'].astype(str)
    df['weather_hour'] = df['Weather'].astype(str)  + '_' + df['hour'].astype(str)
    df['road_hour']    = df['RoadType'].astype(str) + '_' + df['hour'].astype(str)
    df['road_peak']    = df['RoadType'].astype(str) + '_' + df['is_peak_hour'].astype(str)
    df['weather_road'] = df['Weather'].astype(str)  + '_' + df['RoadType'].astype(str)

    return df

train = feature_engineering(train)
test  = feature_engineering(test)

print(f'✅ Feature engineering done')
print(f'   Train shape after FE: {train.shape}')

## 5. Prepare X / y

In [ ]:
train.drop(['Index', 'timestamp'], axis=1, inplace=True)
test.drop(['Index', 'timestamp'],  axis=1, inplace=True)

X      = train.drop('demand', axis=1)
y      = train['demand']
X_test = test.copy()

cat_features = X.select_dtypes(include=['object', 'string']).columns.tolist()
for col in cat_features:
    X[col]      = X[col].fillna('Unknown').astype(str)
    X_test[col] = X_test[col].fillna('Unknown').astype(str)

X      = X.fillna(0)
X_test = X_test.fillna(0)

print(f'Total features  : {X.shape[1]}')
print(f'Categorical     : {cat_features}')

# Optional log transform
y_fit = np.log1p(y) if USE_LOG_TARGET else y.copy()
print(f'\nTarget transform: {"log1p" if USE_LOG_TARGET else "none"}')

## 6. CatBoost — 10-Fold Cross Validation

In [ ]:
CATBOOST_PARAMS = dict(
    iterations          = 8000,
    learning_rate       = 0.01,
    depth               = 10,
    l2_leaf_reg         = 5,
    bagging_temperature = 0.8,
    subsample           = 0.85,
    colsample_bylevel   = 0.75,
    min_data_in_leaf    = 20,
    loss_function       = 'RMSE',
    eval_metric         = 'R2',
    random_seed         = SEED,
    verbose             = 500,
    thread_count        = -1,
)

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof              = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))
scores           = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X)):
    print(f"\n{'='*60}\nFOLD {fold+1}\n{'='*60}")

    X_tr, y_tr = X.iloc[train_idx], y_fit.iloc[train_idx]
    X_vl, y_vl = X.iloc[valid_idx], y_fit.iloc[valid_idx]

    model = CatBoostRegressor(**CATBOOST_PARAMS)
    model.fit(
        X_tr, y_tr,
        cat_features=cat_features,
        eval_set=(X_vl, y_vl),
        use_best_model=True,
        early_stopping_rounds=300,
    )

    vp = model.predict(X_vl)
    oof[valid_idx] = vp

    fold_r2 = r2_score(y_vl, vp)
    print(f'Fold {fold+1} R2 (transformed) = {fold_r2:.6f}')
    scores.append(fold_r2)

    test_predictions += model.predict(X_test) / N_SPLITS

## 7. Results & Evaluation

In [ ]:
# Inverse transform
if USE_LOG_TARGET:
    oof_orig  = np.expm1(oof)
    pred_orig = np.expm1(test_predictions)
else:
    oof_orig  = oof
    pred_orig = test_predictions

oof_orig  = np.clip(oof_orig,  0, None)
pred_orig = np.clip(pred_orig, 0, None)

overall_r2 = r2_score(y, oof_orig)

print('='*60)
print(f'OOF R2 (original scale) : {overall_r2:.6f}')
print(f'Mean Fold R2            : {np.mean(scores):.6f}')
print(f'Std  Fold R2            : {np.std(scores):.6f}')
print('='*60)

# Fold-level bar chart
plt.figure(figsize=(10, 4))
bars = plt.bar(range(1, N_SPLITS+1), scores, color='steelblue', edgecolor='white')
plt.axhline(np.mean(scores), color='coral', linestyle='--', label=f'Mean = {np.mean(scores):.4f}')
plt.axhline(0.98, color='green', linestyle=':', label='Target = 0.98')
plt.xlabel('Fold')
plt.ylabel('R² Score')
plt.title('Per-Fold R² Scores')
plt.legend()
plt.ylim(min(scores) - 0.005, 1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted scatter
sample_idx = np.random.choice(len(y), size=min(3000, len(y)), replace=False)

plt.figure(figsize=(6, 6))
plt.scatter(y.values[sample_idx], oof_orig[sample_idx],
            alpha=0.3, s=5, color='steelblue')
lim = max(y.max(), oof_orig.max())
plt.plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Perfect prediction')
plt.xlabel('Actual demand')
plt.ylabel('Predicted demand')
plt.title(f'Actual vs Predicted  (R² = {overall_r2:.4f})')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
importance = (
    pd.DataFrame({'Feature': X.columns, 'Importance': model.feature_importances_})
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

print('Top 20 Features:')
print(importance.head(20).to_string(index=False))

# Plot top 20
top20 = importance.head(20)
plt.figure(figsize=(10, 6))
plt.barh(top20['Feature'][::-1], top20['Importance'][::-1], color='steelblue')
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances (last fold)')
plt.tight_layout()
plt.show()

## 9. Generate Submission

In [ ]:
submission = pd.DataFrame({
    'Index': submission_ids,
    'demand': pred_orig
})

submission.to_csv('submission_v3.csv', index=False)
print('✅ submission_v3.csv saved!')
print(f'   Rows: {len(submission)}')
print(f'   Demand range: [{pred_orig.min():.2f}, {pred_orig.max():.2f}]')
submission.head(10)